# Phase 3: Which renovations actually cause value?

The question is not "which features correlate with sale price?" Expensive homes tend to have better kitchens, better garages, better basements, and better neighborhoods all at once. Phase 3 asks the causal underwriting question: after controlling for that confounding structure, which renovations actually add value?

## 0. The Question

For a given home, the causal effect of upgrading the kitchen from Fair to Good is the difference between what that same home would sell for with a Good kitchen and what it would sell for with a Fair kitchen, all else equal. We never observe both potential outcomes for the same home. That is the fundamental problem of causal inference. Double Machine Learning is one principled way to estimate this quantity from observational data.

## 1. Why OLS Fails

A naive regression coefficient on `KitchenQual` mixes the kitchen effect with the kind of home and owner that tends to have a better kitchen. `OverallQual` is the running confounder: higher-quality homes have better kitchens and higher prices for many reasons besides the kitchen itself.

```text
OverallQual  ->  KitchenQual
OverallQual  ->  SalePrice
KitchenQual  ->  SalePrice
```

The backdoor path `KitchenQual <- OverallQual -> SalePrice` is what a naive coefficient cannot close unless the confounding structure is explicitly controlled.

## 2. The DML Approach

DML partials out confounders from both the treatment and the outcome using flexible ML models. In this project, LightGBM estimates `E[log1p(SalePrice) | W]` and `E[T | W]`, where `W` is the fixed feature registry plus `OverallQual` and `OverallCond`. The final stage regresses residualized outcome on residualized treatment with HC3 robust standard errors.

The theoretical backbone is the Frisch-Waugh-Lovell theorem: after residualizing both sides on the controls, the treatment coefficient is the same target as the controlled regression coefficient, but DML lets the residualization step be flexible and cross-fitted.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

REPORTS = Path("../reports")
FIGURES = REPORTS / "figures"

effects = pd.read_csv(REPORTS / "phase3_causal_effects.csv")
comparison = pd.read_csv(REPORTS / "phase3_underwriting_comparison.csv")
card = json.loads((REPORTS / "phase3_metric_card.json").read_text())

## 3. Calibration of the Causal Model

The causal layer verifies that prior artifacts exist before estimation: the Phase 1 LightGBM artifact, Phase 2 CQR metric card, Phase 2 calibration curve, and Phase 2 underwriting frame. Cross-fitting is the leakage guard: each residual is produced by nuisance models that did not train on that row's fold.

In [ ]:
card["prior_artifacts"]

## 4. The Comparison Table

A DML coefficient of `$4,450` for `KitchenQual` means: near the median Ames sale price, a one-step kitchen-quality improvement is estimated to cause about `$4,450` in sale-price lift, holding the confounder set fixed. Estimates whose 95% CI crosses zero are statistically inconclusive, not zero.

In [ ]:
columns = [
    "Feature",
    "Naive OLS ($)",
    "DML Causal ($)",
    "DML CI Low ($)",
    "DML CI High ($)",
    "Bias ($)",
    "Statistically Significant?",
]
effects[columns].round(0)

In [ ]:
display(Image(filename=str(FIGURES / "03a_confounding_gap.png")))

## 5. The Renovation Decision Matrix

The matrix compares causal lift to the typical renovation cost assumptions in `config/economics.yaml`. The cost numbers are placeholders, so the matrix is a decision screen rather than an investment recommendation.

In [ ]:
display(Image(filename=str(FIGURES / "03b_renovation_decision_matrix.png")))

## 6. Decision Consequences

The underwriter can now run the prior correlational uplift assumptions and the Phase 3 causal uplift estimates side by side. Under the current 70%-rule purchase model and hard interval-width override, the 10 representative homes produced zero verdict flips. That is still informative: the uncertainty guardrail from Phase 2 dominates these decisions, while causal uplifts move expected profit within the same verdict bands.

In [ ]:
comparison[
    [
        "Id",
        "Neighborhood",
        "correlational_verdict",
        "causal_verdict",
        "correlational_expected_profit",
        "causal_expected_profit",
        "expected_profit_delta",
        "verdict_changed",
    ]
].round(0)

In [ ]:
display(Image(filename=str(FIGURES / "03c_verdict_flip_distributions.png")))

## 7. Limitations and Threats to Validity

The central identifying assumption is conditional independence: after controlling for `W`, treatment assignment is as-good-as-random. This is plausible only to the extent that `W` captures the factors that drive renovation choices: neighborhood, age, size, structural quality, and condition. Remaining threats include unobserved owner wealth, deferred maintenance, contractor quality, permit friction, and micro-location. These are observational estimates, not randomized experimental effects.

## 8. What's Next

Phase 4 asks whether the strategy survives a regime shift. Phase 3 estimates causal renovation effects on pooled cross-sectional Ames data. The next question is whether those effects and the uncertainty-aware underwriting rule would have held through the 2006-2010 housing crash.